# 05 — Hyperparameter Optimization for LightGBM Forecasters

LightGBM has dozens of tunable knobs. The defaults we used in notebooks 03–04
are *reasonable*, not *optimal*. In this notebook we systematically search the
parameter space using two techniques that every practitioner should know:

| Technique | When to use | Pros | Cons |
|---|---|---|---|
| **Grid search** | Small, well-understood spaces (≤ ~30 combos) | Reproducible, exhaustive, easy to explain | Combinatorial blow-up, ignores prior trials |
| **Optuna (TPE)** | Anything bigger; the practical default | Bayesian — learns from history; supports pruning, mixed types, conditional spaces | Stochastic; needs more boilerplate |

### Cross-validation in time-series HPO
We **never** use random k-fold CV on time-series data — it leaks future info into
training. Instead, we use *expanding-window* (or sliding-window) splits: each
fold's validation slice comes strictly after its training slice. The
`utils.cv` module produces these for us.

### What we'll do
1. Build CV folds on the training portion of our data.
2. Run a small **grid search** so we can see the mechanics clearly.
3. Run **Optuna** over a much larger space.
4. Visualise the optimization history.
5. Re-train a final model with the winning configuration.


In [1]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [2]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
DATA_PATH    = "./dataset/m5/m5_tiny.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
EXCLUDE_COLS = []
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon


## 1. Load data and prepare features

We re-use the building blocks from previous notebooks: load → split → feature-engineer.

In [3]:
from utils.data_utils import load_forecasting_data, complete_panel, time_based_split
from utils.feature_engineering import FeatureEngineer

raw = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS, exclude_cols=EXCLUDE_COLS)
panel = complete_panel(raw, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, fill_value=0.0)
cutoff = panel[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(panel, DATE_COL, cutoff=cutoff)

# A small/medium feature set keeps each fold fast — perfect for HPO demos.
fe = FeatureEngineer(date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
                     lags=[1, 7, 14, 28], rolling_windows=[7, 28], ewm_halflives=[7.0, 28.0])

print(f"train rows: {len(train_df):,}   test rows: {len(test_df):,}")


train rows: 220,751   test rows: 14,000


## 2. Build expanding-window CV folds

We carve the training data (only, never the test set) into K folds. For each
fold we hold out the last `valid_days` as the validation slice and use
everything before it for training.


In [4]:
from utils.cv import expanding_window_split, describe_folds
from utils.viz import plot_cv_folds

folds = expanding_window_split(
    train_df, date_col=DATE_COL,
    n_folds=4,
    horizon_days=HOLDOUT_DAYS,   # same horizon as the final test
    gap_days=0,
)
describe_folds(folds)


,fold,train_start,train_end,valid_start,valid_end,n_train,n_valid
0,0,2015-03-08,2016-01-31,2016-02-01,2016-02-28,164751,14000
1,1,2015-03-08,2016-02-28,2016-02-29,2016-03-27,178751,14000
2,2,2015-03-08,2016-03-27,2016-03-28,2016-04-24,192751,14000
3,3,2015-03-08,2016-04-24,2016-04-25,2016-05-22,206751,14000


In [5]:
fig = plot_cv_folds(folds, title="Expanding-window CV folds (train→valid)")
fig.show()


## 3. Grid search

We take a tiny grid (3 × 3 × 2 = 18 combinations). The function evaluates each
combo across all CV folds and returns a sorted DataFrame. The first row is the
winner.


In [6]:
from utils.tuning import grid_search_lgbm
from utils.lgbm_forecaster import DEFAULT_PARAMS
from utils.metrics import wape

base_params = dict(DEFAULT_PARAMS)        # tweedie objective by default

param_grid = {
    "num_leaves":      [31, 63, 127],
    "learning_rate":   [0.03, 0.05, 0.10],
    "min_data_in_leaf":[20, 100],
}

grid_results = grid_search_lgbm(
    train_df, fe, folds,
    base_params=base_params,
    param_grid=param_grid,
    metric_fn=wape,
    num_boost_round=600,
    early_stopping_rounds=50,
    verbose=False,
)
grid_results.head(10)


,trial,num_leaves,learning_rate,min_data_in_leaf,mean_score,std_score,fold0,fold1,fold2,fold3
0,12,127,0.03,20,66.920347,2.156392,70.579999,66.348567,65.613740,65.139081
1,14,127,0.05,20,66.926046,2.016431,70.353510,66.366271,65.705859,65.278543
2,6,63,0.03,20,66.981357,2.167478,70.651738,66.497353,65.456688,65.319648
3,13,127,0.03,100,67.077821,2.348186,71.065580,66.442603,65.146178,65.656924
4,2,31,0.05,20,67.192100,2.124191,70.664217,67.134453,65.768174,65.201554
5,8,63,0.05,20,67.206925,1.944247,70.426773,67.059959,65.599584,65.741382
6,1,31,0.03,100,67.295769,2.336628,71.188983,67.040179,65.457951,65.495961
7,9,63,0.05,100,67.299897,2.227444,71.091159,66.706727,65.758306,65.643395
8,11,63,0.10,100,67.316028,2.168411,70.936938,66.827733,65.224381,66.275060
9,7,63,0.03,100,67.347664,2.192522,71.099735,66.510047,65.572670,66.208203


### Reading the table
- `mean_score` is the mean WAPE across folds — **lower is better**.
- `std_score` tells you how stable the configuration is across folds. A
  configuration that wins one fold but loses the rest is suspicious.
- The fold-by-fold columns let you spot bad luck on a single fold.


In [7]:
import plotly.express as px

# Heat-map of the (num_leaves × learning_rate) plane at the best min_data_in_leaf
best_min_data = grid_results.iloc[0]["min_data_in_leaf"]
sub = grid_results[grid_results["min_data_in_leaf"] == best_min_data]
heat = sub.pivot(index="num_leaves", columns="learning_rate", values="mean_score")
fig = px.imshow(heat, text_auto=".3f", color_continuous_scale="Viridis_r",
                title=f"Grid-search WAPE (min_data_in_leaf={best_min_data})",
                labels={"color": "WAPE"})
fig.show()


## 4. Optuna: Bayesian search with pruning

Optuna's TPE sampler builds a probabilistic model of *good* vs. *bad*
parameters and biases the next trial towards regions that look promising.
We expose a declarative search space via `OptunaSearchSpace`:

- `int` / `float` — continuous or step-wise numeric ranges
- `log_float` — sampled on a log scale (great for `learning_rate`, `lambda_l2`)
- `categorical` — discrete choices

We also enable **pruning** (`MedianPruner`): trials that look worse than the
median after their first fold are aborted, freeing budget for promising
candidates.


In [8]:
from utils.tuning import OptunaSearchSpace, optuna_tune_lgbm

search_space = OptunaSearchSpace({
    "num_leaves":              {"type": "int",       "low": 16, "high": 255, "step": 1},
    "learning_rate":           {"type": "log_float", "low": 1e-3, "high": 3e-1},
    "min_data_in_leaf":        {"type": "int",       "low": 10, "high": 300, "step": 10},
    "feature_fraction":        {"type": "float",     "low": 0.5, "high": 1.0},
    "bagging_fraction":        {"type": "float",     "low": 0.5, "high": 1.0},
    "lambda_l2":               {"type": "log_float", "low": 1e-3, "high": 10.0},
    "tweedie_variance_power":  {"type": "float",     "low": 1.05, "high": 1.95},
})

study, best_params, history = optuna_tune_lgbm(
    train_df, fe, folds,
    base_params=base_params,
    search_space=search_space,
    metric_fn=wape,
    n_trials=30,                  # bump to 100+ for serious runs
    num_boost_round=600,
    early_stopping_rounds=50,
    show_progress_bar=False,
)
print("Best WAPE :", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-05-11 09:22:02,384] A new study created in memory with name: no-name-3cc3094e-e141-48a2-9127-4ee149df42df
[I 2026-05-11 09:22:05,798] Trial 0 finished with value: 69.13992239199 and parameters: {'num_leaves': 105, 'learning_rate': 0.22648248189516848, 'min_data_in_leaf': 220, 'feature_fraction': 0.7993292420985183, 'bagging_fraction': 0.5780093202212182, 'lambda_l2': 0.004207053950287938, 'tweedie_variance_power': 1.1022752509513796}. Best is trial 0 with value: 69.13992239199.
[I 2026-05-11 09:22:17,649] Trial 1 finished with value: 66.85140411819023 and parameters: {'num_leaves': 223, 'learning_rate': 0.030834348179355788, 'min_data_in_leaf': 220, 'feature_fraction': 0.5102922471479012, 'bagging_fraction': 0.9849549260809971, 'lambda_l2': 2.1368329072358767, 'tweedie_variance_power': 1.2411051996104485}. Best is trial 1 with value: 66.85140411819023.
[I 2026-05-11 09:22:32,010] Trial 2 finished with value: 80.80665743187652 and parameters: {'num_leaves': 59, 'learning_rate': 

Best WAPE : 64.97606549778635
Best params:
  num_leaves: 191
  learning_rate: 0.024043577339075195
  min_data_in_leaf: 40
  feature_fraction: 0.8185605796467874
  bagging_fraction: 0.5630745406970603
  lambda_l2: 0.004134437519225516
  tweedie_variance_power: 1.8445064708475225


### Visualising the search
The optimization history shows two things:
- **Trial values** (dots) — how each trial scored.
- **Best-so-far** (line) — the running minimum. A flattening line means
  diminishing returns and is your cue to stop or shrink the budget.

Pruned trials show up with `state == 'PRUNED'` in the history table; they were
short-circuited before they finished all folds.


In [9]:
from utils.viz import plot_optuna_history
fig = plot_optuna_history(history, title="Optuna optimization history (WAPE)")
fig.show()

# Show the trial state distribution
history.groupby("state").size().to_frame("trials")


,trials
state,
COMPLETE,27
PRUNED,3


In [10]:
# Slice plot: each parameter vs. objective
import plotly.express as px
param_cols = [c for c in history.columns if c.startswith("params_")]
melted = history.dropna(subset=["value"]).melt(
    id_vars=["number", "value"], value_vars=param_cols,
    var_name="parameter", value_name="param_value")
fig = px.scatter(
    melted, x="param_value", y="value", color="number",
    facet_col="parameter", facet_col_wrap=3, height=620,
    title="Optuna slice view — objective vs each hyperparameter",
)
fig.update_xaxes(matches=None, showticklabels=True)
fig.update_yaxes(matches=None)
fig.show()


## 5. Train a final model with the winning configuration

`best_params` already includes `base_params` merged with `study.best_params`,
so we can hand it straight to `RecursiveForecaster` and refit on the **full**
training data (no CV split this time — we want every available row).


In [ ]:
from utils.lgbm_forecaster import RecursiveForecaster
from utils.metrics import metric_report

final_model = RecursiveForecaster(feature_engineer=fe, params=best_params,
                                  num_boost_round=2000, early_stopping_rounds=100)
final_model.fit(train_df, valid_df=test_df)   # validation on test for early stop

horizon_dates = sorted(test_df[DATE_COL].unique())
preds = final_model.predict_recursive(history_df=train_df, horizon_dates=horizon_dates)
report = metric_report(test_df[TARGET_COL].values, preds["yhat"].values,
                       y_train=train_df[TARGET_COL].values, season=7)
report


## Recap

- We carved the **training** set into expanding-window CV folds — never letting
  validation data precede training data.
- A small **grid search** is great for explanation and sanity-checks.
- **Optuna** scales to dozens of dimensions cheaply because it learns from
  every trial and prunes the obvious losers.
- We always re-fit the final model on the full training data with the best
  configuration before scoring on the held-out test set.

In **notebook 06** we'll switch gears entirely and meet **Prophet**, an
additive Bayesian-style model that gives us probabilistic forecasts and a
trend/seasonality decomposition out of the box.
